# Load wide CSV exports to Delta

Reads the **manifest**, loads each CSV as **strings** (no schema inference), unions by **column name**, aligns to a **500-column** canonical model, and **appends** to a Delta table.

Run after the generator notebook. Parameters match the generator (bundle variables or widgets).


## Configuration

Set **catalog**, **schema**, and **volume** via the job (bundle) or widgets. Defaults apply when a widget is left empty.


In [ ]:
# Same catalog/schema/volume as the generator (job passes bundle variables into widgets).
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("volume_name", "", "Volume name")


def _param(name: str, default: str) -> str:
    try:
        v = dbutils.widgets.get(name).strip()
        return v if v else default
    except Exception:
        return default


CATALOG = _param("catalog", "main")
SCHEMA = _param("schema", "sf_account_csv_lab")
VOLUME_NAME = _param("volume_name", "sf_account_raw")

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
MANIFEST_PATH = f"{BASE_PATH}/manifest/file_manifest.json"
DELTA_TABLE = f"{CATALOG}.{SCHEMA}.salesforce_account"

_STD_ACCOUNT_FIELDS = (
    "Id Name Type ParentId AccountNumber Site AccountSource AnnualRevenue NumberOfEmployees "
    "Industry Ownership TickerSymbol Rating BillingStreet BillingCity BillingState BillingPostalCode BillingCountry "
    "BillingLatitude BillingLongitude ShippingStreet ShippingCity ShippingState ShippingPostalCode ShippingCountry "
    "Phone Fax Website Description OwnerId CreatedDate LastModifiedDate LastActivityDate IsDeleted PhotoUrl "
    "CleanStatus DunsNumber NaicsCode NaicsDesc SicDesc"
).split()
_CUSTOM_FIELDS = [f"Segment_Score_{i:03d}__c" for i in range(1, 461)]
CANONICAL_COLUMNS = _STD_ACCOUNT_FIELDS + _CUSTOM_FIELDS
assert len(CANONICAL_COLUMNS) == 500

TRUNCATE_TARGET_FIRST = True
DELTA_TARGET_FILES = 16

print("MANIFEST_PATH:", MANIFEST_PATH)
print("DELTA_TABLE:", DELTA_TABLE)


## Optional: truncate target table

Use for repeated test runs; set `TRUNCATE_TARGET_FIRST = False` for append-only accumulation.


In [ ]:
if TRUNCATE_TARGET_FIRST:
    spark.sql(f"DROP TABLE IF EXISTS {DELTA_TABLE}")
    print("Dropped (if existed):", DELTA_TABLE)
else:
    print("Truncate skipped; new rows append to", DELTA_TABLE)


## Load manifest

Reads the JSON manifest from the volume path.


In [ ]:
import json

_stream = dbutils.fs.open(MANIFEST_PATH, "rb")
try:
    raw = _stream.read().decode("utf-8")
finally:
    _stream.close()
manifest = json.loads(raw)
print("files in manifest:", len(manifest))


## Read CSVs and union

**Per file:** `inferSchema=false` keeps reads fast and predictable. **Batched `unionByName`** reduces deep lineage. **Project** fills missing canonical columns with null strings.


In [ ]:
from pyspark.sql import functions as F


def read_one(entry):
    p = entry["volume_path"]
    return (
        spark.read.option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .option("delimiter", ",")
        .csv(p)
    )


def read_and_normalize(entry):
    # Map physical CSV headers back to logical names (blank-header anomaly).
    df = read_one(entry)
    phys = entry.get("columns")
    logical = entry.get("logical_columns")
    if not phys or not logical or len(phys) != len(logical):
        return df
    for p, logical_name in zip(phys, logical):
        if p != logical_name and p in df.columns:
            df = df.withColumnRenamed(p, logical_name)
    return df


def union_batched(entries, batch_size=10):
    batches = []
    for i in range(0, len(entries), batch_size):
        chunk = entries[i : i + batch_size]
        acc = read_and_normalize(chunk[0])
        for e in chunk[1:]:
            acc = acc.unionByName(read_and_normalize(e), allowMissingColumns=True)
        batches.append(acc)
    while len(batches) > 1:
        nxt = []
        for i in range(0, len(batches), batch_size):
            ch = batches[i : i + batch_size]
            a = ch[0]
            for b in ch[1:]:
                a = a.unionByName(b, allowMissingColumns=True)
            nxt.append(a)
        batches = nxt
    return batches[0]


unioned = union_batched(manifest, batch_size=10)

exprs = []
for c in CANONICAL_COLUMNS:
    if c in unioned.columns:
        exprs.append(F.col(c).cast("string").alias(c))
    else:
        exprs.append(F.lit(None).cast("string").alias(c))

final_df = unioned.select(*exprs)
final_df = final_df.repartition(DELTA_TARGET_FILES)

writer = (
    final_df.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
)

writer.saveAsTable(DELTA_TABLE)
print("Appended to", DELTA_TABLE)


## Verify

Row count and sample columns.


In [ ]:
spark.table(DELTA_TABLE).printSchema()
cnt = spark.table(DELTA_TABLE).count()
print("row_count:", cnt)
spark.table(DELTA_TABLE).select("Id", "Name", "BillingCity", "Industry", "Segment_Score_001__c", "Segment_Score_400__c").show(
    5, truncate=False
)
